# Optimization and duality

Two-dimensional linear and quadratic programming by active-boundary enumeration. These routines provide small analytical examples for the numerical checks in notebook 06.


## Numerical scope and interface

The functions `solve_lp_2d` and `solve_qp_2d` operate on two decision variables and enumerate feasible intersections of active constraints. The LP assumes a bounded feasible region; the QP requires a symmetric positive-definite Hessian. The shared helpers construct boundary intersections and test feasibility.


In [ ]:
from itertools import combinations


def _intersection(a, b, c, d):
    determinant = a[0] * c[1] - a[1] * c[0]
    if abs(determinant) < 1e-12:
        return None
    return (
        (b * c[1] - a[1] * d) / determinant,
        (a[0] * d - b * c[0]) / determinant,
    )


def _constraints(A, b, lb, ub):
    if len(A) != len(b):
        raise ValueError("A and b length mismatch.")
    lines = [(tuple(map(float, row)), float(rhs)) for row, rhs in zip(A, b)]
    if any(len(row) != 2 for row, _ in lines):
        raise ValueError("Two decision variables required.")

    for axis in range(2):
        if lb is not None and lb[axis] is not None:
            normal = [0.0, 0.0]
            normal[axis] = -1.0
            lines.append((tuple(normal), -float(lb[axis])))
        if ub is not None and ub[axis] is not None:
            normal = [0.0, 0.0]
            normal[axis] = 1.0
            lines.append((tuple(normal), float(ub[axis])))
    return lines


def _feasible(point, lines, equality, tol=1e-8):
    inequalities = all(
        sum(normal[i] * point[i] for i in (0, 1)) <= rhs + tol
        for normal, rhs in lines
    )
    if equality is None:
        return inequalities
    normal, rhs = equality
    return inequalities and abs(
        sum(normal[i] * point[i] for i in (0, 1)) - rhs
    ) <= tol


def _vertices(lines, equality):
    boundaries = lines + ([equality] if equality is not None else [])
    for (a, b), (c, d) in combinations(boundaries, 2):
        point = _intersection(a, b, c, d)
        if point is not None and _feasible(point, lines, equality):
            yield point


## Linear programming

For $x\in\mathbb{R}^2$, a linear objective $f^\top x$ is minimized subject to $Ax\le b$, optional equalities, and variable bounds. On a nonempty bounded polygon, an optimum occurs at a vertex. The local solver finds intersections of pairs of boundaries, checks feasibility, and evaluates the objective at each candidate.


In [ ]:
def solve_lp_2d(f, A, b, Aeq=None, beq=None, lb=None, ub=None):
    """Minimize a linear objective on a bounded feasible polygon or segment."""
    if len(f) != 2:
        raise ValueError("Exactly two objective coefficients required.")

    equality = None if Aeq is None else (tuple(Aeq), float(beq))
    lines = _constraints(A, b, lb, ub)
    vertices = list(_vertices(lines, equality))
    if not vertices:
        raise ValueError("No feasible vertex (infeasible or unbounded line).")

    optimum = min(vertices, key=lambda x: f[0] * x[0] + f[1] * x[1])
    return optimum, f[0] * optimum[0] + f[1] * optimum[1]


## Quadratic programming

For symmetric positive-definite $H$, the function $\frac12x^\top Hx+f^\top x$ has positive curvature. A constrained optimum may occur at the unconstrained stationary point, on an active edge, or at a vertex. The two-dimensional solver evaluates those cases explicitly.


In [ ]:
def solve_qp_2d(H, f, A, b, Aeq=None, beq=None, lb=None, ub=None):
    """Minimize a strictly convex quadratic in two variables."""
    if len(H) != 2 or any(len(row) != 2 for row in H) or len(f) != 2:
        raise ValueError("H must be 2x2 and f must contain two coefficients.")

    a, off_diagonal, c = H[0][0], H[0][1], H[1][1]
    determinant = a * c - off_diagonal ** 2
    if abs(H[1][0] - off_diagonal) > 1e-10 or a <= 0 or determinant <= 1e-12:
        raise ValueError("H must be symmetric positive definite.")

    lines = _constraints(A, b, lb, ub)
    equality = None if Aeq is None else (tuple(Aeq), float(beq))
    boundaries = [equality] if equality is not None else lines
    candidates = list(_vertices(lines, equality))

    if equality is None:
        candidates.append((
            (-c * f[0] + off_diagonal * f[1]) / determinant,
            (off_diagonal * f[0] - a * f[1]) / determinant,
        ))

    for normal, rhs in boundaries:
        norm_squared = normal[0] ** 2 + normal[1] ** 2
        if norm_squared <= 1e-15:
            continue

        point = (normal[0] * rhs / norm_squared,
                 normal[1] * rhs / norm_squared)
        direction = (-normal[1], normal[0])
        H_direction = (
            a * direction[0] + off_diagonal * direction[1],
            off_diagonal * direction[0] + c * direction[1],
        )
        H_point = (
            a * point[0] + off_diagonal * point[1],
            off_diagonal * point[0] + c * point[1],
        )
        numerator = -sum(direction[i] * (H_point[i] + f[i]) for i in (0, 1))
        denominator = sum(direction[i] * H_direction[i] for i in (0, 1))
        tau = numerator / denominator
        candidates.append((
            point[0] + tau * direction[0],
            point[1] + tau * direction[1],
        ))

    candidates = [x for x in candidates if _feasible(x, lines, equality)]
    if not candidates:
        raise ValueError("No feasible candidate.")

    def objective(x):
        quadratic = (
            a * x[0] ** 2
            + 2 * off_diagonal * x[0] * x[1]
            + c * x[1] ** 2
        )
        return 0.5 * quadratic + f[0] * x[0] + f[1] * x[1]

    optimum = min(candidates, key=objective)
    return optimum, objective(optimum)
